In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator as Estimator
from scipy.optimize import minimize
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [ ]:
SEED = 42
backend = AerSimulator(seed_simulator=SEED)
sampler = Sampler(backend)

# Variational Quantum Eigensolver

The Variational Quantum Eigensolver (VQE) is a hybrid quantum-classical algorithm whose objective is to approximate the ground state of a given Hamiltonian $H$. Hence, its goal is to find a state $|\psi\rangle$ such that:

$$
E_{\psi} = \langle \psi |H|\psi\rangle
$$

is minimized.

It is also possible to estimate this minimum using other algorithms, such as some using Quantum Phase Estimation. The main advantage of VQE over those approaches is that it is better suited to today's NISQ devices, since it uses relatively shallow circuits and does not require fault-tolerant quantum computers.

The VQE is based on the Variational Principle, which states that for any normalized state $|\psi\rangle$, the expected value of the Hamiltonian is always greater than or equal to the true ground-state energy $E_0$:

$$
\langle \psi | H | \psi \rangle \geq E_0
$$

where $E_0$ is the lowest eigenvalue of the Hamiltonian $H$. Equality is only reached when $|\psi\rangle$ is the true ground state of the system.

This principle is the theoretical basis of the VQE algorithm. Instead of trying to directly compute the exact ground state, VQE prepares a family of trial quantum states $|\psi(\theta)\rangle$ using a parametrized quantum circuit. Then, the algorithm searches for the parameters $\theta$ that minimize the expectation value of the Hamiltonian. The closer the ansatz can get to the true ground state, the closer the estimated energy will be to the real minimum energy of the system.

A condition for VQE is that $H$ must be able to be expressed as a linear combination of tensor products of Pauli matrices. This decomposition is important because the expectation value of the Hamiltonian can then be estimated by measuring each Pauli term separately and combining the results. Luckily, this is always the case for observables in finite Hilbert spaces, which are exactly the ones we work with in quantum computing. Given that, the algorithm is quite simple:

- Design a parametrized quantum circuit $V(\theta)$, usually called ansatz.
- With the help of a quantum computer, use the ansatz to prepare the quantum state $|\psi(\theta)\rangle$.
- Measure the prepared state and estimate the energy $E_{\psi(\theta)}$.
- Use a classical optimizer to change the ansatz's parameters in order to minimize the energy. 
- Repeat this optimization step until some stopping criterion is met and the optimal parameters $\theta_0$ are found.

Once the algorithm has finished, we can use $\theta_0$ to both find the state of minimum energy:

$$
|\psi(\theta_0)\rangle = V(\theta_0)|0\rangle
$$

and to find the value of the minimum energy:

$$
E_0 = E_{\psi(\theta_0)} = \langle \psi(\theta_0) |H|\psi(\theta_0)\rangle
$$

The choice of the ansatz in VQE is a crucial factor in the algorithm's performance. For this reason, it is important to incorporate domain-specific knowledge of the problem into the design of the ansatz. At the same time, the ansatz should remain shallow and easy to implement, since deep circuits are more affected by noise in current NISQ devices.

# Finding excited stated using VQE

With a small modification, we can use VQE to find excited states of $H$. First we need to use standard VQE to find a ground state $|\psi(\theta_0)\rangle = V(\theta_0)|0\rangle$ with energy $\lambda_0$. Then, we design the excited Hamiltonian:

$$
H' = H + C |\psi_0 \rangle \langle \psi_0 |
$$

where C is a positive real number.

We can take any normalized state $|\psi \rangle$ and compute the expectation value:

$$
\langle H'\rangle_{\psi} = \langle \psi |H'|\psi\rangle 
= 
\langle \psi |H|\psi\rangle
+
C \langle \psi | \psi_0 \rangle \langle \psi_0 | \psi \rangle
=
\langle \psi |H|\psi\rangle
+
C |\langle \psi_0 | \psi \rangle|^2
$$

We can see that the expectation value of $H'$ is the expectation value of $H$ plus a non-negative value that quantifies the overlap of $| \psi \rangle$ and $| \psi_0 \rangle$. That way, we have two extreme cases for that term: if $| \psi \rangle$ = $| \psi_0  \rangle$ that term will be $C$ and if $| \psi \rangle$ and $| \psi_0 \rangle$ are orthogonal, that term will be 0.

It can easily be proven that if $\lambda_0 \leq \lambda_1 \leq ... \lambda_n$ are the eigenvalues of H associated to each eigenvector $\{|\lambda_j \rangle \}$, then:
$$
\langle \lambda_j |H'| \lambda_j \rangle = \lambda_j
$$

when $j \neq 0$ and:

$$
\langle \lambda_0 |H'| \lambda_0 \rangle = C + \lambda_0
$$

Hence if $C>\lambda_1 - \lambda_0$, then $ |\psi_0 \rangle = |\lambda_0\rangle $ will no longer be a ground state of $H'$ because the energy of $|\lambda_1\rangle $ will be lower that of $|\psi_0\rangle $ and consequently, $|\lambda_1\rangle $ must be a ground state of $H'$.

We can use this method to obtain any excited state, not just the first one. For example to find the second excited state, you can consider the Hamiltonian:

$$
H'' = H' + C' |\lambda_1 \rangle \langle \lambda_1 |
$$

and use VQE to obtain $|\lambda_2\rangle$. Exactly as before, it is essential to select an adequate value of $C'$ for this process.

Note that, in order to calculate $|\langle \psi_0 |\psi (\theta) \rangle |^2 $ using a quantum computer, we will take advantage of the fact that:

$$
|\langle \psi_0 |\psi (\theta) \rangle |^2 = |\langle 0|V(\theta_0)^\dagger V(\theta_0)|0\rangle |^2 
$$

Which simply is the probability of obtaining $|0\rangle$ when applying consecutively our ansantz $V$ with $\theta$ as parameters and the inverse of our ansantz $V^\dagger$ with $\theta_0$ as parameters to the state $|0\rangle$.